# 01 — RNN Limitations & Why Attention Matters
### Solution Notebook

This notebook contains completed solutions for every exercise in the starter notebook.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Part A — Minimal RNN (unchanged)

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn_cell = nn.RNNCell(hidden_size, hidden_size)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        batch, seq_len = x.shape
        embeds = self.embedding(x)
        h = torch.zeros(batch, self.hidden_size, device=x.device)
        outputs = []
        for t in range(seq_len):
            h = self.rnn_cell(embeds[:, t], h)
            outputs.append(self.output(h))
        return torch.stack(outputs, dim=1)


rnn = SimpleRNN(vocab_size=100, hidden_size=64).to(device)
x_demo = torch.randint(0, 100, (4, 20), device=device)
print('RNN output shape:', rnn(x_demo).shape)

## Part B — Vanishing Gradient (solution)

In [ ]:
def measure_gradient_flow(model, seq_lengths, vocab_size=100):
    grad_norms = []
    for seq_len in seq_lengths:
        model.zero_grad()
        x = torch.randint(0, vocab_size, (1, seq_len), device=device)
        targets = torch.randint(0, vocab_size, (1, seq_len), device=device)

        # Solution B1: forward pass + loss + backward
        logits = model(x)                        # (1, seq_len, vocab_size)
        loss = F.cross_entropy(
            logits.view(-1, vocab_size),
            targets.view(-1)
        )
        loss.backward()

        # Gradient norm at the embedding layer
        grad_norm = model.embedding.weight.grad.norm().item()
        grad_norms.append(grad_norm)

    return grad_norms


# Solution B2: LSTM
class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm_cell = nn.LSTMCell(hidden_size, hidden_size)  # ← added
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        batch, seq_len = x.shape
        embeds = self.embedding(x)
        h = torch.zeros(batch, self.hidden_size, device=x.device)
        c = torch.zeros(batch, self.hidden_size, device=x.device)
        outputs = []
        for t in range(seq_len):
            h, c = self.lstm_cell(embeds[:, t], (h, c))
            outputs.append(self.output(h))
        return torch.stack(outputs, dim=1)


seq_lengths = [5, 10, 20, 50, 100, 200]
lstm = SimpleLSTM(vocab_size=100, hidden_size=64).to(device)

rnn_grads  = measure_gradient_flow(rnn,  seq_lengths)
lstm_grads = measure_gradient_flow(lstm, seq_lengths)

plt.figure(figsize=(8, 4))
plt.plot(seq_lengths, rnn_grads,  'o-', color='tomato',    label='RNN')
plt.plot(seq_lengths, lstm_grads, 's-', color='steelblue', label='LSTM')
plt.xlabel('Sequence length')
plt.ylabel('Embedding gradient norm')
plt.title('Vanishing gradients: RNN vs LSTM')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nObservation: RNN gradients collapse exponentially.')
print('LSTM is better but still degrades — the fundamental issue is sequential depth.')

## Part C — Sequential Bottleneck (unchanged, works as-is)

In [ ]:
def time_rnn(seq_len, hidden=128, n_runs=20):
    rnn_bench = nn.RNN(hidden, hidden, batch_first=True).to(device)
    x = torch.randn(8, seq_len, hidden, device=device)
    with torch.no_grad(): rnn_bench(x)
    if device.type == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs): rnn_bench(x)
    if device.type == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - start) / n_runs * 1000

def time_matmul(seq_len, hidden=128, n_runs=20):
    x = torch.randn(8, seq_len, hidden, device=device)
    W = torch.randn(hidden, hidden, device=device)
    if device.type == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs): x @ W
    if device.type == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - start) / n_runs * 1000

lengths = [32, 64, 128, 256, 512]
rnn_times    = [time_rnn(l)    for l in lengths]
matmul_times = [time_matmul(l) for l in lengths]

plt.figure(figsize=(8, 4))
plt.plot(lengths, rnn_times,    'o-', color='tomato',  label='RNN (sequential)')
plt.plot(lengths, matmul_times, 's-', color='seagreen', label='MatMul (parallel)')
plt.xlabel('Sequence length'); plt.ylabel('Time (ms)')
plt.title('Sequential vs Parallel Computation'); plt.legend()
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## Part D — Long-Range Copy Task (solution)

In [ ]:
from src.models.attention import MultiHeadAttention

VOCAB = 10

def make_copy_task_batch(batch_size, seq_len, vocab_size=VOCAB):
    x = torch.randint(1, vocab_size, (batch_size, seq_len))
    target = x[:, 0].clone()
    return x.to(device), target.to(device)


# ---- tiny attention model ----
class TinyAttentionModel(nn.Module):
    """Embedding + one attention layer + linear output."""
    def __init__(self, vocab_size, d_model=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.attn  = MultiHeadAttention(d_model, n_heads=4)
        self.fc    = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        e, _ = self.attn(self.embed(x), self.embed(x), self.embed(x))
        return self.fc(e[:, -1])  # predict from last position


def train_copy_task(model, seq_len, epochs=200, lr=1e-3, batch=32):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        x, y = make_copy_task_batch(batch, seq_len)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
    # Eval
    model.eval()
    with torch.no_grad():
        x, y = make_copy_task_batch(256, seq_len)
        acc = (model(x).argmax(-1) == y).float().mean().item()
    model.train()
    return acc


# RNN wrapper that predicts from last hidden state
class RNNCopyModel(nn.Module):
    def __init__(self, vocab_size, hidden=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden)
        self.rnn   = nn.RNN(hidden, hidden, batch_first=True)
        self.fc    = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        out, _ = self.rnn(self.embed(x))
        return self.fc(out[:, -1])


test_lengths = [10, 30, 50]
print(f'{'Length':>8}  {'RNN acc':>10}  {'Attn acc':>10}')
print('-' * 35)
for sl in test_lengths:
    rnn_m  = RNNCopyModel(VOCAB).to(device)
    attn_m = TinyAttentionModel(VOCAB).to(device)
    rnn_acc  = train_copy_task(rnn_m,  sl, epochs=300)
    attn_acc = train_copy_task(attn_m, sl, epochs=300)
    print(f'{sl:>8}  {rnn_acc:>10.2%}  {attn_acc:>10.2%}')

## Summary

The results above illustrate three concrete failure modes of RNNs:

1. **Vanishing gradients** — gradient norms collapse exponentially with depth for RNNs; LSTMs help but don't fully solve it.
2. **Sequential bottleneck** — wall-clock time grows nearly linearly with sequence length; parallel operations do not.
3. **Long-range copy task** — RNN accuracy falls as the dependency distance grows; a single attention layer maintains high accuracy regardless.

These motivate the **Transformer** — which we build from scratch starting in the next section.

**Next:** `../part2_fundamentals/02_attention_mechanism_starter.ipynb`